In [ ]:
# Data manipulation libraries
import pandas as pd
import numpy as np
from scipy import stats

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# To ensure that plots are displayed inline in jupyter notebook
%matplotlib inline

# remove warnings
import warnings
warnings.simplefilter('ignore')

print('Successfully Import Libraries')


Successfully Import Libraries


In [ ]:
import pandas as pd
import kagglehub

# 1. Download the latest version
path = kagglehub.dataset_download("zusmani/pakistans-largest-ecommerce-dataset")

# 2. Point directly to the CSV file
# kagglehub.dataset_download returns the folder path, so we join it with the filename
import os
file_name = "Pakistan Largest Ecommerce Dataset.csv"
full_path = os.path.join(path, file_name)

# 3. Load with robust settings
df = pd.read_csv(
    full_path,
    encoding='latin1',
    low_memory=False,
    on_bad_lines='skip' # This skips the rows that are causing the "saw 3 fields" error
)

# 4. Immediate Cleanup: Drop those completely empty columns we saw earlier
df = df.dropna(axis=1, how='all')

# 5. Let's see the result
print(f"Dataset Loaded! Shape: {df.shape}")
df.head()

Using Colab cache for faster access to the 'pakistans-largest-ecommerce-dataset' dataset.
Dataset Loaded! Shape: (1048575, 21)


,item_id,status,created_at,sku,price,qty_ordered,grand_total,increment_id,category_name_1,sales_commission_code,...,payment_method,Working Date,BI Status,MV,Year,Month,Customer Since,M-Y,FY,Customer ID
0,211131.0,complete,7/1/2016,kreations_YI 06-L,1950.0,1.0,1950.0,100147443,Women's Fashion,\N,...,cod,7/1/2016,#REF!,"1,950",2016.0,7.0,2016-7,7-2016,FY17,1.0
1,211133.0,canceled,7/1/2016,kcc_Buy 2 Frey Air Freshener & Get 1 Kasual Bo...,240.0,1.0,240.0,100147444,Beauty & Grooming,\N,...,cod,7/1/2016,Gross,240,2016.0,7.0,2016-7,7-2016,FY17,2.0
2,211134.0,canceled,7/1/2016,Ego_UP0017-999-MR0,2450.0,1.0,2450.0,100147445,Women's Fashion,\N,...,cod,7/1/2016,Gross,"2,450",2016.0,7.0,2016-7,7-2016,FY17,3.0
3,211135.0,complete,7/1/2016,kcc_krone deal,360.0,1.0,60.0,100147446,Beauty & Grooming,R-FSD-52352,...,cod,7/1/2016,Net,360,2016.0,7.0,2016-7,7-2016,FY17,4.0
4,211136.0,order_refunded,7/1/2016,BK7010400AG,555.0,2.0,1110.0,100147447,Soghaat,\N,...,cod,7/1/2016,Valid,"1,110",2016.0,7.0,2016-7,7-2016,FY17,5.0


In [ ]:


# 1. Drop completely empty columns (Unnamed: 21 to 25)
df = df.drop(columns=[col for col in df.columns if 'Unnamed' in col])

# 2. Drop the "ghost rows" (where item_id is NaN)
# This will bring your row count from 1.04M down to exactly 584,524
df = df.dropna(subset=['item_id'])

# 3. Correct Data Types
# item_id and Customer ID should be integers (but can't be if they have NaNs,
# so fill NaNs with 0 or drop them first)
df['Customer ID'] = df['Customer ID'].fillna(0).astype(int)
df['item_id'] = df['item_id'].astype(int)

# 4. Clean the " MV " column (notice the leading space in your info printout)
df.rename(columns={' MV ': 'mv'}, inplace=True)
# If MV has currency symbols, strip them:
# df['mv'] = df['mv'].replace('[\$,]', '', regex=True).astype(float)

# 5. Standardize Dates
df['created_at'] = pd.to_datetime(df['created_at'])
df['Working Date'] = pd.to_datetime(df['Working Date'])

print(df.info())

<class 'pandas.core.frame.DataFrame'>
Index: 584524 entries, 0 to 584523
Data columns (total 21 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   item_id                584524 non-null  int64         
 1   status                 584509 non-null  object        
 2   created_at             584524 non-null  datetime64[ns]
 3   sku                    584504 non-null  object        
 4   price                  584524 non-null  float64       
 5   qty_ordered            584524 non-null  float64       
 6   grand_total            584524 non-null  float64       
 7   increment_id           584524 non-null  object        
 8   category_name_1        584360 non-null  object        
 9   sales_commission_code  447346 non-null  object        
 10  discount_amount        584524 non-null  float64       
 11  payment_method         584524 non-null  object        
 12  Working Date           584524 non-null  datetime6

In [ ]:
# Strip any leading/trailing spaces from string columns
df['status'] = df['status'].str.strip().str.lower()
df['category_name_1'] = df['category_name_1'].str.strip()

# Handle Financial Noise: We only want transactions that make sense
# Filtering for positive prices and grand_totals
df = df[(df['price'] >= 0) & (df['qty_ordered'] > 0)]

# Handle 'status' noise: Standardizing for easier SQL analysis
# We group many messy statuses into 4 main categories
status_map = {
    'complete': 'Completed', 'closed': 'Completed', 'received': 'Completed', 'paid': 'Completed',
    'canceled': 'Canceled', 'order_refunded': 'Refunded', 'refund': 'Refunded',
    'pending': 'Pending', 'holded': 'Pending', 'processing': 'Pending'
}
df['status_clean'] = df['status'].map(status_map).fillna('Others')

# Drop columns that are redundant for a SQL Star Schema (like Year, Month, M-Y, FY)
# We will regenerate these using SQL/DAX later for better practice
cols_to_drop = ['Year', 'Month', 'M-Y', 'FY', 'BI Status', 'sales_commission_code']
df_clean = df.drop(columns=cols_to_drop)

In [ ]:
df.describe()

,item_id,created_at,price,qty_ordered,grand_total,discount_amount,Working Date,Year,Month,Customer ID
count,584524.000000,584524,5.845240e+05,584524.000000,5.845240e+05,584524.000000,584524,584524.000000,584524.000000,584524.000000
mean,565667.074218,2017-08-08 11:42:53.589450752,6.348748e+03,1.296388,8.530619e+03,499.492775,2017-08-08 11:42:53.589450752,2017.044115,7.167654,45789.650245
min,211131.000000,2016-07-01 00:00:00,0.000000e+00,1.000000,-1.594000e+03,-599.500000,2016-07-01 00:00:00,2016.000000,1.000000,0.000000
25%,395000.750000,2017-01-29 00:00:00,3.600000e+02,1.000000,9.450000e+02,0.000000,2017-01-29 00:00:00,2017.000000,4.000000,13516.000000
50%,568424.500000,2017-08-17 00:00:00,8.990000e+02,1.000000,1.960400e+03,0.000000,2017-08-17 00:00:00,2017.000000,7.000000,42855.000000
75%,739106.250000,2018-02-03 00:00:00,4.070000e+03,1.000000,6.999000e+03,160.500000,2018-02-03 00:00:00,2018.000000,11.000000,73535.000000
max,905208.000000,2018-08-28 00:00:00,1.012626e+06,1000.000000,1.788800e+07,90300.000000,2018-08-28 00:00:00,2018.000000,12.000000,115326.000000
std,200121.173648,NaN,1.494927e+04,3.996061,6.132081e+04,1506.943046,NaN,0.707355,3.486305,34415.211831


In [ ]:
# 1. Clean 'mv' (Market Value) - It was an object/string, needs to be float
# Remove any non-numeric characters (like commas or spaces) if they exist
df['mv'] = pd.to_numeric(df['mv'], errors='coerce').fillna(0)

# 2. Fill the last few Nulls
# 20 rows are missing SKU and 164 are missing Category
df['sku'] = df['sku'].fillna('Unknown_SKU')
df['category_name_1'] = df['category_name_1'].fillna('Unknown_Category')

In [ ]:
# 1. Filter out Customer ID 0 if you want clean user analytics
# Or keep it, but realize it represents 'Guest'
df = df[df['Customer ID'] > 0]

# 2. Convert Customer ID to Int one last time (just to be safe)
df['Customer ID'] = df['Customer ID'].astype(int)

In [ ]:
# I will drop the 'sales_commission_code' column due to over 50% missing values, as it is not suitable for filling due to its uniqueness
df.drop('sales_commission_code', axis=1, inplace=True)
# We will remove the 'M-Y' column from the dataset, as its information is already represented in the separate 'Year' and 'Month' columns
df.drop('M-Y', axis=1, inplace=True)

print("Successfully dropped 'M-Y' & 'sales_commission_code' columns from the dataset")

Successfully dropped 'M-Y' & 'sales_commission_code' columns from the dataset


In [ ]:
# Rename columns with meaningful names
df.rename(columns={
    ' MV ': 'market_value',
    'FY': 'fiscal_year',
    'Year': 'year',
    'Month': 'month',
    'sku': 'stock_keeping_unit',
    'Customer Since': 'customer_since',
    'Customer ID': 'customer_id',
    'category_name_1': 'category_name',
    'BI Status': 'bi_status',
    'Working Date': 'working_date'
}, inplace=True)

print("rename columns successfully")

rename columns successfully


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 584513 entries, 0 to 584523
Data columns (total 20 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   item_id             584513 non-null  int64         
 1   status              584498 non-null  object        
 2   created_at          584513 non-null  datetime64[ns]
 3   stock_keeping_unit  584513 non-null  object        
 4   price               584513 non-null  float64       
 5   qty_ordered         584513 non-null  float64       
 6   grand_total         584513 non-null  float64       
 7   increment_id        584513 non-null  object        
 8   category_name       584513 non-null  object        
 9   discount_amount     584513 non-null  float64       
 10  payment_method      584513 non-null  object        
 11  working_date        584513 non-null  datetime64[ns]
 12  bi_status           584513 non-null  object        
 13  mv                  584513 non-nul

In [ ]:
# 1. Dim_Customer: Descriptive data about users
dim_customer = df[['customer_id', 'customer_since']].drop_duplicates(subset=['customer_id'])

# 2. Dim_Product: Descriptive data about items
dim_product = df[['stock_keeping_unit', 'category_name']].drop_duplicates(subset=['stock_keeping_unit'])

# 3. Fact_Sales: The quantitative transactions
# We remove descriptive columns like category_name because they now live in Dim_Product
fact_sales = df[[
    'item_id', 'customer_id', 'stock_keeping_unit', 'status_clean',
    'price', 'qty_ordered', 'grand_total', 'discount_amount',
    'payment_method', 'created_at', 'working_date'
]]

In [ ]:
from google.colab import files

# Save your Star Schema tables to CSV
dim_customer.to_csv('dim_customer.csv', index=False)
dim_product.to_csv('dim_product.csv', index=False)
fact_sales.to_csv('fact_sales.csv', index=False)

# Download them to your computer
files.download('dim_customer.csv')
files.download('dim_product.csv')
files.download('fact_sales.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Save using a Pipe (|) instead of a Comma (,)
dim_customer.to_csv('dim_customer_v3.csv', sep='|', index=False)
dim_product.to_csv('dim_product_v3.csv', sep='|', index=False)
fact_sales.to_csv('fact_sales_v3.csv', sep='|', index=False)

from google.colab import files
files.download('dim_customer_v3.csv')
files.download('dim_product_v3.csv')
files.download('fact_sales_v3.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>